#`Evaluation`

1. The Quality Metrics (LLM-as-a-Judge)
  * Relevance: Does the report actually answer the user's specific constraints?

  * Hallucination Check: Did the Analyst cite real facts, or did it invent data?

  * Adversarial Success: Did the Synthesis actually solve the risks raised by the Devil's Advocate?

  * Feasibility: Is the technical implementation physically/digitally possible?

2. The Diversity Metrics (Mathematical)
We don't want 5 ideas that are basically the same.

Semantic Distance: We convert the 5 generated ideas into "Vectors" (Embeddings) and calculate the distance between them.

  * Low Score: The ideas are synonyms (Bad).

  * High Score: The ideas are distinct approaches (Good).

3. The Consistency Metrics (Structural)
Schema Compliance: Did the JSON Architect output valid JSON?

Latency: Time taken per agent

In [11]:
!pip install -q --force-reinstall transformers peft accelerate sentence-transformers && pip install -q -U crewai litellm langchain-community python-dotenv scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 112.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 488.0/488.0 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 132.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [12]:
import os
import json
from google.colab import userdata
from crewai import Agent, Task, Crew, LLM
from pydantic import BaseModel, Field
from typing import List

In [13]:
api_key = userdata.get('OPENROUTER_API_KEY')
os.environ['OPENROUTER_API_KEY'] = api_key

In [14]:


# --- 1. DATA SCHEMA (The Target Format) ---
class IdeaOption(BaseModel):
    title: str = Field(..., description="The name of the idea")
    description: str = Field(..., description="The summary/description of the idea")
    risk: str = Field(..., description="The main risk associated with this idea")

class IdeationReport(BaseModel):
    summary: str = Field(..., description="The market summary found in the text")
    options: List[IdeaOption] = Field(..., description="The list of 5 options found in the text")

# --- 2. CONVERTER FUNCTION ---
def convert_markdown_to_structure(markdown_text):
    """
    Reads raw Markdown and uses an LLM to extract structured JSON.
    """
    # Use a cheap, fast model for extraction (e.g., GPT-4o-mini)
    extractor_llm = LLM(
        model="openrouter/openai/gpt-4o-mini",
        api_key=os.environ.get("OPENROUTER_API_KEY"),
        temperature=0.0
    )

    extractor = Agent(
        role="Data Extractor",
        goal="Extract structured data from the provided text report.",
        backstory="Expert at parsing unstructured reports into strict JSON formats.",
        llm=extractor_llm,
        verbose=True
    )

    extraction_task = Task(
        description=f"""
        Read the following report and extract the Market Summary and the 5 Idea Options into JSON.

        REPORT TEXT:
        {markdown_text}
        """,
        expected_output="A structured JSON object.",
        agent=extractor,
        output_pydantic=IdeationReport # <--- This enforces the structure
    )

    crew = Crew(agents=[extractor], tasks=[extraction_task])
    result = crew.kickoff()

    # Returns the Pydantic object
    return result.pydantic

In [15]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# --- A. METRICS ---
class ScoreCard(BaseModel):
    novelty_score: int = Field(..., description="1-10: How unique are these ideas?")
    feasibility_score: int = Field(..., description="1-10: Are these realistic?")
    risk_handling_score: int = Field(..., description="1-10: Did the synthesis address the Devil's critiques?")
    market_alignment_score: int = Field(..., description="1-10: Do these fit the Analyst's market data?")
    critique: str = Field(..., description="A short paragraph explaining the scores.")

def calculate_diversity_score(options):
    try:
        model = SentenceTransformer('all-MiniLM-L6-v2')
        embeddings = model.encode(options)
        similarity_matrix = cosine_similarity(embeddings)
        upper_tri = similarity_matrix[np.triu_indices(len(embeddings), k=1)]
        if len(upper_tri) == 0: return 0.0
        avg_similarity = np.mean(upper_tri)
        return round((1 - avg_similarity) * 10, 2)
    except Exception:
        return 0.0

# --- B. JUDGE FUNCTION ---
def run_evaluation(structured_data):
    # 1. Prepare Data
    data = structured_data.dict()
    summary = data.get("summary", "")
    options = data.get("options", [])
    titles = [opt['title'] for opt in options]

    # 2. Math Metric
    div_score = calculate_diversity_score(titles)

    # 3. AI Judge
    judge_llm = LLM(
        model="openrouter/openai/gpt-4o",
        api_key=os.environ.get("OPENROUTER_API_KEY"),
        temperature=0.1
    )

    evaluator = Agent(
        role="Impartial Judge",
        goal="Grade the output objectively based on a strict rubric and provide clear justifications.",
        backstory="Strict VC evaluator, known for meticulous and unbiased scoring based on clear, detailed rubrics, and always providing explicit justifications for assigned scores.",
        llm=judge_llm
    )

    eval_task = Task(
        description=f"""
        Evaluate these ideas based on the summary provided, using the following detailed rubric for each score of 1-10.
        When scoring, consider the definitions for 1, 5, and 10 as benchmarks and interpolate for other scores.
        In your `critique`, you MUST explain your reasoning for each score, explicitly referencing aspects of the ideas and how they align (or don't align) with the given rubric definitions for 1, 5, or 10, or interpolated values. This justification is crucial for understanding the specific score given.

        **REPORT SUMMARY:**
        {summary}

        **IDEA OPTIONS:**
        {json.dumps(options, indent=2)}

        **SCORING RUBRIC:**

        **1. Novelty (1-10): How unique and innovative are these ideas compared to existing solutions?**
        *   **1 - Very Low Novelty:** Ideas are direct copies or minor variations of widely available solutions. No new insights or approaches.
        *   **5 - Moderate Novelty:** Ideas show some differentiation but are based on established concepts or technologies. They offer incremental improvements.
        *   **10 - Very High Novelty:** Ideas are truly groundbreaking, introducing entirely new concepts, technologies, or market approaches. They challenge existing paradigms.

        **2. Feasibility (1-10): Are these realistic and practical to implement given current technology, resources, and market conditions?**
        *   **1 - Very Low Feasibility:** Ideas are highly theoretical, require unproven technology, or face insurmountable regulatory/resource hurdles. Implementation seems impossible.
        *   **5 - Moderate Feasibility:** Ideas are implementable but require significant R&D, substantial investment, or face notable technical/logistical challenges.
        *   **10 - Very High Feasibility:** Ideas are well within current technological capabilities and resource availability. Implementation path is clear and achievable with reasonable effort.

        **3. Risk Handling (1-10): How well do the ideas acknowledge and address potential risks (e.g., privacy, ethical, market, technical) raised by critical analysis?**
        *   **1 - Very Poor Risk Handling:** Ideas show no awareness of significant risks, or proposed mitigations are entirely inadequate and superficial. Critical flaws unaddressed.
        *   **5 - Moderate Risk Handling:** Ideas acknowledge key risks but proposed solutions are generic, partially developed, or might not fully address the severity of the risks.
        *   **10 - Excellent Risk Handling:** Ideas comprehensively identify all major risks and propose robust, innovative, and practical strategies for mitigation or circumvention. Risks are proactively managed.

        **4. Market Alignment (1-10): Do these ideas fit the Analyst's market data and current market trends, and address a real market need?**
        *   **1 - Very Low Market Alignment:** Ideas are misaligned with market data, address a non-existent or declining need, or target a shrinking market segment. No clear demand.
        *   **5 - Moderate Market Alignment:** Ideas have some connection to market trends but might target a niche audience, face strong competition, or have an unclear path to market adoption.
        *   **10 - Excellent Market Alignment:** Ideas strongly resonate with identified market needs and trends, target a large and growing segment, and offer a compelling solution with clear market demand.

        Your output MUST be a JSON ScoreCard object.
        """,
        expected_output="JSON ScoreCard",
        agent=evaluator,
        output_pydantic=ScoreCard
    )

    crew = Crew(agents=[evaluator], tasks=[eval_task])
    result = crew.kickoff()
    scores = result.pydantic

    return scores, div_score

RuntimeError: Failed to import transformers.trainer because of the following error (look up to see its traceback):
cannot import name 'set_rng_state_for_device' from 'transformers.trainer_pt_utils' (/usr/local/lib/python3.12/dist-packages/transformers/trainer_pt_utils.py)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 1. Define your file path (Upload your markdown file to Colab first)
file_path = "/content/drive/MyDrive/Copy of final_report (9).md" # Corrected path
if os.path.exists(file_path):
    print("📂 Reading Markdown file...")
    with open(file_path, "r") as f:
        markdown_text = f.read()

    print("🔄 Converting Markdown to JSON (Structure Architect)...")
    # Step A: Convert
    structured_data = convert_markdown_to_structure(markdown_text)

    print("⚖️ Running Evaluation...")
    # Step B: Evaluate
    scores, div_score = run_evaluation(structured_data)

    # Step C: Print Report
    print(f"""
    \n\n==========================================
    📊 POST-MORTEM EVALUATION REPORT
    ==========================================

    [1] DIVERSITY SCORE: {div_score}/10

    [2] QUALITY SCORES:
    - Novelty:      {scores.novelty_score}/10
    - Feasibility:  {scores.feasibility_score}/10
    - Risk Mgmt:    {scores.risk_handling_score}/10
    - Alignment:    {scores.market_alignment_score}/10

    [3] CRITIQUE:
    "{scores.critique}"
    ==========================================
    """)
else:
    print(f"❌ Error: Could not find {file_path}. Please upload the file.")

📂 Reading Markdown file...
🔄 Converting Markdown to JSON (Structure Architect)...


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Data Extractor                                                                                          │
│                                                                                                                 │
│  Task:                                                                                                          │
│          Read the following report and extract the Market Summary and the 5 Idea Options into JSON.             │
│                                                                                                                 │
│          REPORT TEXT:                                                                                           │
│          # Executive Summary — Assessment: Webcam-Based Gaze & Emotion Tracking with Automatic Pay Docking      │
│                                                                                                                 │
│  Short framing (So what?)                                                                                       │
│  - The proposed product—using laptop webcams to track gaze and emotional states and automatically docking pay   │
│  if an employee looks away >5 minutes/hour—is commercially plausible but ethically, legally, and operationally  │
│  high-risk. It trades short-term monitoring gains for severe reputational, compliance, and workforce-cost       │
│  risks.                                                                                                         │
│  - Practical alternative: pivot toward ethically defensible, high-value features (employee-owned insights,      │
│  accessibility, contextual analytics) that deliver productivity value without coercive surveillance. The        │
│  portfolio below evaluates five refined options and gives clear action recommendations.                         │
│                                                                                                                 │
│  Top-line recommendation                                                                                        │
│  - Do not pursue the pay-docking webcam surveillance as described. Instead, prioritize Option 2                 │
│  (Employee-Owned Performance Insights) and Option 5 (Workplace Accessibility & Accommodation Platform) in       │
│  parallel. Treat Option 1 (Integrated Project Intelligence) as a conditional enterprise bet if strict privacy   │
│  and partner commitments are secured. Hold Options 3 and 4 for very specific, governed customers.               │
│                                                                                                                 │
│  Options — concise, action-oriented briefs                                                                      │
│                                                                                                                 │
│  Option 1 — Integrated Project Intelligence Platform (Contextual Productivity Insights)                         │
│  - What: Middleware analytics copilot that explains WHY metrics move across Jira/Asana/Monday. SaaS pricing     │
│  $150–$400/employee/year; platform revenue-share distribution.                                                  │
│  - So what?: High enterprise demand if positioned as explanatory (not punitive). Major revenue potential but    │
│  risks of sliding into individual surveillance.                                                                 │
│  - Ethics: Moderate concern — surveillance and bias risks. Mitigations: data minimization, short retention      │
│  (90–180 days), human-in-the-loop review, transparency dashboards, contractual limits.                          │
│  - Tech feasibility: High (8/10). 12–18 months to MVP; 

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Data Extractor                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "summary": "The proposed product—using laptop webcams to track gaze and emotional states and automatically   │
│  docking pay if an employee looks away >5 minutes/hour—is commercially plausible but ethically, legally, and    │
│  operationally high-risk. It trades short-term monitoring gains for severe reputational, compliance, and        │
│  workforce-cost risks. Practical alternative: pivot toward ethically defensible, high-value features            │
│  (employee-owned insights, accessibility, contextual analytics) that deliver productivity value without         │
│  coercive surveillance. The portfolio below evaluates five refined options and gives clear action               │
│  recommendations.",                                                                                             │
│    "options": [                                                                                                 │
│      {                                                                                                          │
│        "title": "Integrated Project Intelligence Platform (Contextual Productivity Insights)",                  │
│        "description": "Middleware analytics copilot that explains WHY metrics move across Jira/Asana/Monday.    │
│  SaaS pricing $150–$400/employee/year; platform revenue-share distribution.",                                   │
│        "risk": "Moderate concern — surveillance and bias risks."                                                │
│      },                                                                                                         │
│      {                                                                                                          │
│        "title": "Employee-Owned Performance Insights (Selective Sharing)",                                      │
│        "description": "Freemium, employee-controlled app collecting behavioral metadata (calendar, email        │
│  metadata, meeting density). Optional, consented anonymized sharing to managers; enterprise upsell for team     │
│  features.",                                                                                                    │
│        "risk": "Low–moderate concern."                                                                          │
│      },                                                                                                         │
│      {                                                                                                          │
│        "title": "Insider Threat Prevention for High-Compliance Verticals",                                      │
│        "description": "Endpoint + SIEM-integrated insider-threat detection for regulated sectors (healthcare,   │
│  finance, government).",                                                                                        │
│        "risk": "High concern."                                                                                  │
│      },                                                                                                         │
│      {                                                                                                          │
│        "title": "Resource Allocation Optimization Engin

⚖️ Running Evaluation...


NameError: name 'run_evaluation' is not defined

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 

